# HGF Quickstart: From Simulation to Parameter Recovery

This notebook walks through a complete Hierarchical Gaussian Filter (HGF) analysis
using **Variational Laplace** — the same fitting algorithm as the MATLAB HGF Toolbox
(TAPAS `tapas_fitModel`), reimplemented in Python/JAX.

We follow the "Ten Simple Rules for Computational Modeling" framework
(Wilson & Collins, 2019, *eLife*) applied to the HGF:

| Step | Wilson & Collins Rule | What we do |
|------|---------------------|------------|
| 1 | Design the experiment | Load the 3-cue PRL task config |
| 2 | Design the model | Understand the 2-level and 3-level binary HGF |
| 3 | Simulate | Generate synthetic data with known parameters |
| 4 | Fit parameters | VB-Laplace (MAP + Hessian) |
| 5 | Parameter recovery | Can we get the true parameters back? |
| 6 | Model comparison | 2-level vs 3-level via Laplace model evidence + BMS |

**Prerequisites:** `pip install -e .` (or `pip install -e ".[dev]"` for plots)

**Runtime:** ~2 minutes on CPU with the small demo cohort.

### Related notebooks

- **`parameter_explorer.ipynb`** — Interactive slider dashboard to see how each HGF parameter affects belief trajectories and choice probabilities
- **`design_experiment.ipynb`** — Power analysis: find the N and effect size needed for your experiment before collecting data

In [ ]:
from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

---
## Step 1: Design the Experiment

We use a **3-cue probabilistic reversal learning (PRL)** task:

- 3 cues with different reward probabilities (e.g., 0.8, 0.5, 0.2)
- Participant chooses one cue per trial and receives binary reward
- **Partial feedback**: only the chosen cue's outcome is observed
- Reward probabilities reverse periodically (testing adaptability)

All task parameters are defined in a single YAML config file — never hardcoded.

In [ ]:
from prl_hgf.env.task_config import load_config

config = load_config()  # loads configs/prl_analysis.yaml
print(f"Task:              {config.task.name}")
print(f"N per group:       {config.simulation.n_participants_per_group}")
print(f"Groups:            {list(config.simulation.groups.keys())}")
print(f"Trials per session:{config.task.n_trials_total}")

# Session structure (from session_deltas)
group0 = list(config.simulation.session_deltas.keys())[0]
session_labels = ["baseline"] + config.simulation.session_deltas[group0].session_labels
print(f"Sessions:          {session_labels}")

---
## Step 2: Design the Model

The HGF has two components (just like TAPAS):

### Perceptual model (HGF)
A hierarchy of Gaussian random walks that track hidden environmental states:

- **Level 1** ($x_1$): Reward probability per cue (binary, via sigmoid of $x_2$)
- **Level 2** ($x_2$): Log-odds of reward — a random walk with volatility controlled by level 3
- **Level 3** ($x_3$, 3-level only): Meta-volatility — tracks *how quickly* the environment is changing

### Response model (softmax + stickiness)

$$P(\text{choose cue } k) = \text{softmax}(\beta \cdot \hat{p}_k + \zeta \cdot \mathbb{1}[\text{prev} = k])$$

### Key parameters

| Parameter | Symbol | Meaning | Identifiable? |
|-----------|--------|---------|---------------|
| Tonic volatility | $\omega_2$ | Baseline learning rate (log-space) | Yes |
| Inverse temperature | $\beta$ | Decision noise (higher = more deterministic) | Yes |
| Stickiness | $\zeta$ | Choice perseveration (+ = sticky, - = switching) | Yes |
| Meta-volatility | $\omega_3$ | Volatility of volatility (3-level only) | Poor |
| Coupling | $\kappa$ | How much $x_3$ modulates learning rate | Frozen at 1.0 |

---
## Step 3: Simulate Synthetic Data

We generate a cohort of synthetic participants with **known ground-truth parameters**.
This is the foundation for validating the fitting pipeline.

For this quickstart we use a small cohort (baseline session only, 10 participants)
to keep runtime under 2 minutes.

In [ ]:
from prl_hgf.simulation.batch import simulate_batch

# Simulate full cohort, then filter for speed
sim_df = simulate_batch(config)
sim_df = sim_df[sim_df["session"] == "baseline"].copy()
pids = sorted(sim_df["participant_id"].unique())[:10]
sim_df = sim_df[sim_df["participant_id"].isin(pids)].copy()

print(f"Participants: {sim_df['participant_id'].nunique()}")
print(f"Trials per participant: {len(sim_df) // sim_df['participant_id'].nunique()}")
sim_df.head()

### Inspect ground-truth parameters

Each participant has known `true_*` parameters that we'll try to recover.

In [ ]:
true_params = sim_df.groupby("participant_id")[
    ["true_omega_2", "true_beta", "true_zeta", "true_omega_3", "true_kappa"]
].first()
true_params.style.format("{:.3f}")

### Visualise one agent's task structure

In [ ]:
pid_example = pids[0]
agent_df = sim_df[sim_df["participant_id"] == pid_example].copy()

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# True reward probabilities
for i in range(3):
    axes[0].plot(agent_df["trial"].values, agent_df[f"cue_{i}_prob"].values,
                 label=f"Cue {i}", alpha=0.8)
axes[0].set_ylabel("True P(reward)")
axes[0].legend(loc="upper right")
axes[0].set_title(f"Participant {pid_example}")

# Choices and rewards
colors = ["C0", "C1", "C2"]
for _, row in agent_df.iterrows():
    marker = "o" if row["reward"] == 1 else "x"
    axes[1].scatter(row["trial"], row["cue_chosen"],
                    c=colors[int(row["cue_chosen"])], marker=marker,
                    s=10, alpha=0.6)
axes[1].set_ylabel("Cue chosen")
axes[1].set_xlabel("Trial")
axes[1].set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()

---
## Step 4: Fit Parameters with VB-Laplace

VB-Laplace (Variational Bayes / Laplace approximation) is the default fitting
method in the MATLAB HGF Toolbox. It works by:

1. **MAP optimisation** — find the posterior mode using quasi-Newton (L-BFGS)
2. **Hessian at the mode** — compute the curvature via automatic differentiation
3. **Gaussian approximation** — the posterior is approximately $\mathcal{N}(\hat{\theta}, H^{-1})$

This is much faster than MCMC (seconds vs minutes per participant) and gives
comparable point estimates for well-identified parameters.

| Method | MATLAB (TAPAS) | This toolbox |
|--------|---------------|---------------|
| Fit function | `tapas_fitModel` | `fit_vb_laplace_prl()` |
| Optimiser | `fminunc` / quasi-Newton | `jaxopt.LBFGS` (JAX-compiled) |
| Hessian | Finite differences | `jax.hessian` (exact autodiff) |
| Output | `est` struct | `arviz.InferenceData` |
| Model evidence | `est.optim.LME` | `idata.attrs['lme']` |

In [ ]:
from prl_hgf.fitting.fit_vb_laplace_prl import fit_vb_laplace_prl

# Fit 3-level HGF
t0 = time.perf_counter()
idata_3level = fit_vb_laplace_prl(sim_df, model_name="hgf_3level", n_pseudo_draws=500)
print(f"3-level fit: {time.perf_counter() - t0:.1f}s")

# Fit 2-level HGF
t0 = time.perf_counter()
idata_2level = fit_vb_laplace_prl(sim_df, model_name="hgf_2level", n_pseudo_draws=500)
print(f"2-level fit: {time.perf_counter() - t0:.1f}s")

In [ ]:
# Inspect the 3-level posterior
print("Posterior variables:", list(idata_3level.posterior.data_vars))
print("Dims:", dict(idata_3level.posterior.dims))
az.summary(idata_3level, var_names=["omega_2", "beta", "zeta", "omega_3"])

---
## Step 5: Parameter Recovery

The critical validation step: correlate **fitted** parameter estimates with
**true** values from simulation. Good recovery ($r \geq 0.7$) means the
model can reliably distinguish individual differences.

**Known caveat**: $\omega_3$ (meta-volatility) recovery is poor in the
literature. This is not a bug — it reflects genuine identifiability limits
of the 3-level HGF with typical trial counts.

In [ ]:
from prl_hgf.fitting.fit_vb_laplace_prl import idata_to_fit_df
from prl_hgf.analysis.recovery import build_recovery_df, compute_recovery_metrics

# Convert InferenceData -> long-form fit DataFrame
fit_df = idata_to_fit_df(idata_3level, ["omega_2", "beta", "zeta", "omega_3"])

# Join with true parameters and compute metrics
recovery_df = build_recovery_df(sim_df, fit_df, min_n=0)
metrics_df = compute_recovery_metrics(recovery_df)

print("Recovery metrics (3-level HGF):")
print("=" * 55)
for _, row in metrics_df.iterrows():
    status = "PASS" if row["r"] >= 0.7 else "FAIL"
    print(f"  {row['parameter']:>10s}:  r = {row['r']:.3f}  "
          f"bias = {row['bias']:+.3f}  rmse = {row['rmse']:.3f}  [{status}]")
print("=" * 55)

In [ ]:
from prl_hgf.analysis.plots import plot_recovery_scatter

fig = plot_recovery_scatter(recovery_df, metrics_df)
plt.tight_layout()
plt.show()

---
## Step 6: Model Comparison

We compare 2-level vs 3-level HGF using **random-effects Bayesian Model
Selection** (Rigoux et al. 2014) — the same framework as TAPAS
`spm_BMS` / `tapas_bayesian_model_selection`.

The Laplace approximation provides per-participant **log model evidence
(LME)** — the marginal likelihood of the data under each model, penalised
for complexity.  This is stored as `idata.attrs['lme']` (equivalent to
TAPAS `est.optim.LME`).

Since the data was generated from a 3-level process, the 3-level model
should win.

In [ ]:
from prl_hgf.fitting.fit_vb_laplace_prl import compare_models_laplace

bms = compare_models_laplace({
    "3-level HGF": idata_3level,
    "2-level HGF": idata_2level,
})

print("Log Model Evidence (LME) summary:")
print(bms["lme_summary"].to_string(index=False))
print()
print(f"Exceedance probability (xp):           "
      f"{dict(zip(bms['model_names'], np.round(bms['xp'], 3)))}")
print(f"Protected exceedance probability (pxp): "
      f"{dict(zip(bms['model_names'], np.round(bms['pxp'], 3)))}")
print(f"Bayesian Omnibus Risk (BOR):             {bms['bor']:.4f}")

In [ ]:
# Visualise per-participant LME difference
lme_diff = np.array(idata_3level.attrs["lme"]) - np.array(idata_2level.attrs["lme"])

fig, ax = plt.subplots(figsize=(8, 3))
colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in lme_diff]
ax.bar(range(len(lme_diff)), lme_diff, color=colors, alpha=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Participant")
ax.set_ylabel("LME(3-level) - LME(2-level)")
ax.set_title("Per-participant model preference (green = 3-level wins)")
plt.tight_layout()
plt.show()

---
## Summary

You've completed the core HGF analysis workflow:

1. **Simulated** a cohort with known parameters from `configs/prl_analysis.yaml`
2. **Fitted** both 2-level and 3-level HGF using VB-Laplace (seconds, not hours)
3. **Recovered** ground-truth parameters and assessed identifiability
4. **Compared** models using Laplace log model evidence + random-effects BMS

### MATLAB TAPAS equivalences

| TAPAS | Python equivalent |
|-------|-------------------|
| `tapas_simModel` | `simulate_batch(config)` |
| `tapas_fitModel` | `fit_vb_laplace_prl(sim_df, model_name)` |
| `est.optim.LME` | `idata.attrs['lme']` |
| `spm_BMS` / `tapas_bayesian_model_selection` | `compare_models_laplace(idata_dict)` |
| `tapas_fit_plotCorr` | `plot_recovery_scatter(recovery_df, metrics_df)` |

### Next steps

- **Increase N** for stable recovery: change `[:10]` filter to `[:60]` or remove it
- **Full pipeline**: run `python scripts/demo_quickstart.py` (or `make demo`)
- **MCMC fitting**: use `fit_batch_hierarchical()` for full posterior (requires GPU/cluster)
- **Real data**: replace `simulate_batch()` with your data loader
- **Different task**: swap `load_config()` for `load_pat_rl_config()` (PAT-RL task)